## Working with Messaging Topics

# Mastering Google Cloud Pub/Sub: Core Topics and Fan-Out Architecture

Hello, learners! In today's lesson, we'll delve deeper into messaging concepts on Google Cloud. One of the core building blocks in Google Cloud messaging is the **topic**. Topics act as central communication channels where messages are published, ensuring that all attached subscriptions receive those records seamlessly.

Our focus today will be on understanding topics and how they enable reliable, asynchronous, and scalable communication between different decoupled layers of your application infrastructure.

---

## Understanding Topics and Subscriptions

In Google Cloud messaging, topics are central to the Publish/Subscribe (Pub/Sub) model. A topic is a named gateway resource to which messages are sent by publishers. Subscriptions, conversely, are independent entities attached to a specific topic that pull or receive those exact message streams.

### Key Specifications & Safeguards

* **Default Scale Limits:** You can create up to **10,000 topics per project** by default (this soft quota can be easily increased via support requests).
* **Massive Throughput:** Each topic is backed by Google's global infrastructure mesh, natively supporting **millions of messages per second** without pre-provisioning.
* **Payload Constraints:** The absolute maximum size limit for an individual message payload is **10 MB**.
* **Decoupled Delivery Architecture:** Each individual topic handles multiple concurrent subscriptions, and each subscription independently routes messages to separate endpoint configurations.

### Types of Subscriptions and Fallbacks

* **Pull Subscriptions:** Your client consumer application takes the driver's seat, explicitly checking for and extracting messages from the buffer queue.
* **Push Subscriptions:** The Pub/Sub router takes the lead, automatically delivering inbound message payloads directly to a secured, publicly accessible HTTP or HTTPS webhook endpoint.
* **Dead-Letter Topics:** Subscriptions can be configured to isolate un-processable or continuously crashing payloads, forwarding them automatically to a separate quarantine topic for deep analysis or manual reprocessing.

Each individual subscription handles its own delivery retry logic, data persistence boundaries, and acknowledgment windows independently to suit your specific downstream runtime dependencies.

---

## Push vs. Pull Subscriptions: Architectural Trade-offs

Choosing the correct consumption topography depends heavily on your networking footprint and how much control you want over resource execution scaling:

| Operational Factor | Pull Subscription | Push Subscription |
| --- | --- | --- |
| **Latency** | Slightly higher (driven by client polling intervals) | **Ultra-low** (messages are forced downstream immediately) |
| **Scaling Control** | **Client Managed:** Scale up manually by spinning up more container workers | **Platform Managed:** GCP automatically scales out webhook hits |
| **Endpoint Management** | **Private:** No public URL or ingress rule required; client initiates outgoing gRPC sockets | **Public:** Requires a secure, publicly discoverable, valid HTTPS certificate endpoint |
| **Retry Strategy** | Handled natively via client loop code configuration and lease modifications | Handled automatically by GCP infrastructure using back-off retry matrices |
| **Primary Use Cases** | Heavy batch jobs, long-running database operations, systems behind strict corporate firewalls | Real-time user notifications, instant webhook transformations, serverless functions (Cloud Run) |

* **When to lean towards Pull:** Choose pull whenever your consuming microservices live inside a private network, demand high processing times, or if you need absolute control over throttling intake speeds.
* **When to lean towards Push:** Choose push when you are integrating directly with public external APIs, building event-driven webhooks, or prioritizing immediate, real-time message delivery to lightweight compute targets.

### Real-World Architectural Example: E-Commerce Funnel

Imagine a single user clicks "Place Order" in an e-commerce checkout loop. The frontend pushes a token payload to an `order-updates` topic. From there, multiple backend services scale out in parallel to execute their tasks concurrently:

```
                  ┌──> [ Pull Subscription ] ──> Inventory System (Checks warehouse stocks)
                  │
[ Order Topic ] ──┼──> [ Push Subscription ] ──> Notification Engine (Fires SMS/Email alerts)
                  │
                  └──> [ Pull Subscription ] ──> BigQuery Analytics Pipeline (Logs behavioral charts)

```

---

## Creating Topics via the Python SDK

Let's transition to implementing this programmatically. Creating an ingestion point using the official `google-cloud-pubsub` client library requires minimal boilerplate code.

```python
from google.cloud import pubsub_v1

# 1. Initialize the monolithic network Publisher client
publisher = pubsub_v1.PublisherClient()

# 2. Specify target project coordinate credentials
project_id = "your-project-id"
topic_id = "my-topic"

# 3. Construct the fully qualified regional/global location path representation
# Result: "projects/your-project-id/topics/my-topic"
topic_path = publisher.topic_path(project_id, topic_id)

# 4. Fire the gRPC request to provision the resource block
topic = publisher.create_topic(request={"name": topic_path})

# 5. Output confirmation validation
print("Created topic:", topic.name)

```

**Console Output:**

> `Created topic: projects/your-project-id/topics/my-topic`

---

## Configuring Advanced Topic Attributes

After provisioning a topic, you can dynamically configure its metadata to optimize organizational visibility, management control, and resource cost tracking.

### Assigning Categorization Labels

Labels are structural key-value string pairs injected directly into your resource metadata layer. They are extremely valuable for isolating billing costs or distinguishing deployment footprints.

```python
# Identify target tracking metadata
labels = {"env": "production", "team": "analytics"}

# Build updates using an explicit field mask
publisher.update_topic(
    request={
        "topic": {"name": topic_path, "labels": labels},
        "update_mask": {"paths": ["labels"]}  # Informs the API server to alter ONLY this property block
    }
)

print("Labels set for topic:", labels)

```

**Console Output:**

> `Labels set for topic: {'env': 'production', 'team': 'analytics'}`

> 🔒 **Security Access Control Note:** To lock down who can publish payloads or subscribe to this stream, you can query and adjust Identity and Access Management (IAM) policies via the GCP console or the underlying API client, granting explicit roles like `roles/pubsub.publisher` to dedicated service accounts.

---

## Inspecting Project Infrastructures: Listing Active Topics

When managing a growing multi-service cluster, your tools must be able to inventory existing routes. Here is how you cleanly scan all active endpoints within your project scope:

### 1. Basic Name Scans

```python
# Format the project scope string container matching API specs
project_scope = f"projects/{project_id}"

print("Discovered Topics:")
for topic in publisher.list_topics(request={"project": project_scope}):
    print(topic.name)

```

**Console Output:**

> `projects/your-project-id/topics/my-topic`
> `projects/your-project-id/topics/order-updates`
> `projects/your-project-id/topics/user-events`
> `projects/your-project-id/topics/system-alerts`

### 2. Deep Object Inspection (Extracting Configurations)

If you want to view structural metadata properties alongside your inventory list, query the properties nested inside each item wrapper:

```python
for topic in publisher.list_topics(request={"project": f"projects/{project_id}"}):
    print(f"Topic Path: {topic.name}")
    print(f"  └─ Labels: {dict(topic.labels)}")
    print(f"  └─ Storage Policy Settings: {topic.message_storage_policy.allowed_persistence_regions}")
    print("-" * 40)

```

**Console Output:**

```text
Topic Path: projects/your-project-id/topics/my-topic
  └─ Labels: {'env': 'production', 'team': 'analytics'}
  └─ Storage Policy Settings: []
----------------------------------------
Topic Path: projects/your-project-id/topics/order-updates
  └─ Labels: {'env': 'production', 'service': 'ecommerce'}
  └─ Storage Policy Settings: []
----------------------------------------

```

---

## Achieving Fan-Out and Reliable Delivery

Google Cloud Pub/Sub implements the structural **fan-out** messaging pattern seamlessly. Because subscription tracking data models remain independent of one another, adding, modifying, or deleting consumer applications has **zero performance impact** on your core ingestion topic pipeline.

Messages persist securely in the storage layer and will not disappear until every single individual consumer endpoint successfully registers its acknowledgment, ensuring reliable delivery and concurrent event execution at enterprise scale.

---

## Review, Practice, and Next Steps

Let's review the milestones we hit today:

* Analyzed the structural relationship between Pub/Sub **topics** and attached **subscriptions**.
* Identified when to implement lightweight **Push** webhooks versus resilient, deep-cutting **Pull** processing loops.
* Programmatically created, labeled, and inventoried cloud topics from scratch using the Google Python SDK framework.

In your upcoming lab assignments, you'll put this foundational knowledge into action by building your own operational topic topologies, applying custom labels, and executing cluster scans inside the interactive training simulator. Happy coding!

## Google Cloud Pub/Sub Topic Creation with Python

In this task, you will write a Python script that involves Google Cloud Pub/Sub. A pre-written script, which, when executed, creates a Pub/Sub topic named 'MyTopic' and assigns labels to it, will be provided. Additionally, after creating the Pub/Sub topic, the script will list all existing Pub/Sub topics to ensure that your new topic has been created successfully. There is no need for code modification in this task; your role is simply to execute the script and observe the topic creation process in Google Cloud Pub/Sub. Pay close attention to how the google-cloud-pubsub client is initialized, how a new topic is created, how labels are assigned to this topic, and finally, how the script lists all the Pub/Sub topics. Be aware that there is a pre-existing topic named 'MyHeartbeat' in our environment. However, this topic might take a bit of time to become available. Depending on when you click the "Run" button, you may or may not see two topics in the script output.

Important Note: Executing scripts can modify the resources in our GCP simulator. To return to the initial state, you can use the reset button located in the top right corner. However, be aware that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard

```python
from google.cloud import pubsub_v1

# Initialize Pub/Sub publisher client
publisher = pubsub_v1.PublisherClient()

# Set project ID (in practice, this would be set via environment variables or service account)
project_id = "your-project-id"

# Create topic path
topic_path = publisher.topic_path(project_id, 'MyTopic')

# Create a new Pub/Sub topic with labels (equivalent to display name attribute)
topic = publisher.create_topic(
    request={
        "name": topic_path,
        "labels": {
            "display_name": "my_topic_display_name"
        }
    }
)

# Confirm the Pub/Sub topic creation
print("Created Pub/Sub topic:", topic.name)

# List all Pub/Sub topics
project_path = f"projects/{project_id}"
topics = publisher.list_topics(request={"project": project_path})
print("List of Pub/Sub Topics:", [topic.name for topic in topics])

```

## Update Google Cloud Pub/Sub Topic Display Name Label

Building further upon Google Cloud Pub/Sub, you will now have the task of extending a provided Python script. The initial part of the script creates a Google Cloud Pub/Sub topic named 'MyTopic'. Your duty is to extend this script by adding additional functionality that sets a display_name label for the created Pub/Sub topic using the topic path. As you've learned in the lessons, labels in Pub/Sub are key-value pairs that provide metadata for topics. Upon successful execution, the script should print out the topic name and the successful setting of the display_name label.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import pubsub_v1

# Initialize Pub/Sub publisher client
publisher = pubsub_v1.PublisherClient()

# Set project ID (in practice, this would be set via environment variables or service account)
project_id = "your-project-id"

# Create topic path
topic_path = publisher.topic_path(project_id, 'MyTopic')

# TODO: Create a new Pub/Sub topic with a display_name label

```

To set a label on a Google Cloud Pub/Sub topic during its creation, you pass the `labels` dictionary as part of the `topic` configuration object inside your `create_topic` request.

Here is the completed script that handles the creation of the topic while successfully applying your custom `display_name` metadata label:

```python
from google.cloud import pubsub_v1

# Initialize Pub/Sub publisher client
publisher = pubsub_v1.PublisherClient()

# Set project ID (in practice, this would be set via environment variables or service account)
project_id = "your-project-id"

# Create topic path
topic_path = publisher.topic_path(project_id, 'MyTopic')

# --- TODO FIXED: Create a new Pub/Sub topic with a display_name label ---

# Define the label metadata key-value pair
topic_metadata = {
    "name": topic_path,
    "labels": {
        "display_name": "mytopic-production-feed"  # Adding the requested display_name label
    }
}

# Provision the topic along with its embedded labels in a single call
topic = publisher.create_topic(request=topic_metadata)

# Print verification output
print("Topic created successfully:", topic.name)
print("Successfully set display_name label:", topic.labels.get("display_name"))

```

---

### Key Takeaway on Topic Labels

Labels are simple key-value pairs (`dict` structures) that attach lightweight, indexable metadata directly to your GCP infrastructure components.

Applying tags like `display_name` helps team members instantly identify what a resource does when scanning the infrastructure, and it allows DevOps pipelines to filter, group, or organize resource tracking metrics across massive project deployment environments.

## Listing Google Cloud Pub/Sub Topics

## Create Pub/Sub Topic and Set Labels